# 05 — Length Extrapolation Lab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/marcoharuni/forge-position/blob/main/notebooks/05_length_extrapolation_lab.ipynb)

Run a tiny decoder-only Transformer on a synthetic next-token task and compare positional methods at longer lengths.

**Runtime:** <10 minutes on free Colab T4

---

## 1. Install & Clone

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
!git clone https://github.com/marcoharuni/forge-position.git
%cd forge-position
!/root/.local/bin/uv sync

## 2. Imports

In [ ]:
import sys
sys.path.insert(0, 'src')

import torch
import torch.nn.functional as F

from forge_position import TinyDecoderOnlyTransformer, TinyTransformerConfig

print(torch.__version__)
print(torch.cuda.is_available())

## 3. Synthetic sequence task

Each sequence is a modular counting pattern. This is a sanity check, not a benchmark.

In [ ]:
def make_batch(batch_size, seq_len, vocab_size):
    starts = torch.arange(batch_size).unsqueeze(1) % vocab_size
    return (starts + torch.arange(seq_len).unsqueeze(0)) % vocab_size

make_batch(3, 8, 16)

## 4. Model runner

In [ ]:
def run_method(method, train_length=16, eval_length=32):
    torch.manual_seed(42)
    vocab_size = 24
    model = TinyDecoderOnlyTransformer(TinyTransformerConfig(
        vocab_size=vocab_size,
        d_model=24,
        n_heads=4,
        n_layers=1,
        max_seq_len=max(train_length, eval_length),
        position_method=method,
    ))
    opt = torch.optim.AdamW(model.parameters(), lr=4e-3)
    for _ in range(6):
        tokens = make_batch(8, train_length, vocab_size)
        logits, _ = model(tokens)
        loss = F.cross_entropy(logits[:, :-1].reshape(-1, vocab_size), tokens[:, 1:].reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        tokens = make_batch(8, eval_length, vocab_size)
        logits, _ = model(tokens)
        loss = F.cross_entropy(logits[:, :-1].reshape(-1, vocab_size), tokens[:, 1:].reshape(-1))
        acc = (logits[:, :-1].argmax(dim=-1) == tokens[:, 1:]).float().mean().item()
    return float(loss), acc

## 5. Compare methods

In [ ]:
methods = ['learned', 'sinusoidal', 'rope', 'alibi', 'nope']
for method in methods:
    loss, acc = run_method(method)
    print(f'{method:12s} loss={loss:.4f} accuracy={acc:.3f}')

---

**Done.** For a script version, run `uv run python examples/07_compare_methods.py` from the repository root.